In [22]:
import pandas as pd
import numpy as np
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
from preprocessing import get_features_and_target
from visualizer import plot_visualizer
import plotly.graph_objects as go
from tabpfn import TabPFNRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor

In [23]:
import huggingface_hub
huggingface_hub.login()

# Getting Dataframe

In [24]:
# Load the training and development datasets
train_df = pd.read_csv("data/train_data.csv")
dev_df = pd.read_csv("data/development_data.csv")
sc = StandardScaler()

target_column = "PullTest (N)" 

x_train, y_train = get_features_and_target(train_df, target_column)
x_dev, y_dev = get_features_and_target(dev_df, target_column)

x_train_scale = sc.fit_transform(X=x_train)
x_dev_scale = sc.transform(x_dev)


In [25]:
train_df.shape[0]

295

# Defining Model

In [26]:
#model = 'XGBoost'
#model = 'RandomForest'
model = 'TabPFN'

# Fit Model

In [27]:
if model == 'TabPFN':

    # Initialize the regressor
    regressor = TabPFNRegressor()  # Uses TabPFN-2.5 weights, trained on synthetic data only.
    # To use TabPFN v2:
    # regressor = TabPFNRegressor.create_default_for_version(ModelVersion.V2)
    regressor.fit(x_train_scale, y_train)

    # Predict on the test set
    predictions = regressor.predict(x_dev_scale)

elif model == 'XGBoost':

    # Convert the data into DMatrix format
    dtrain = xgb.DMatrix(x_train_scale, label=y_train)
    dtest = xgb.DMatrix(x_dev_scale, label=y_dev)

    # Set the parameters for the XGBoost model
    params = {
        'objective': 'reg:squarederror',
        'max_depth': 5,
        'eta': 0.3,
        'eval_metric': 'rmse',
    }

    # Train the model
    num_boost_round = 5
    bst = xgb.train(params, dtrain, num_boost_round)

    # Make predictions
    predictions = bst.predict(dtest)

elif model == 'RandomForest':

    # Set the parameters for the Random Forest model
    params = {
            'n_estimators': 11,
            'max_depth': 5,
            'min_samples_split': 2,
            'min_samples_leaf': 4,
            'random_state': 42,
    }

    predictions = RandomForestRegressor(**params).fit(x_train_scale, y_train).predict(x_dev_scale)


In [28]:
predictions

array([4255.312 , 2165.7305, 4269.653 , 2165.2998, 5169.084 , 2253.5068,
       5122.244 , 2261.5337, 3386.4512, 3295.1057, 3336.8684, 2851.0684,
       2862.5144, 2866.2256, 2835.9568, 2843.328 , 2819.895 , 2848.7856,
       2732.5054, 2787.107 , 2741.9116, 2748.6377, 2731.4995, 2723.731 ,
       2730.2493, 2776.7437, 2737.186 , 2747.7095, 2745.2231, 2740.5933,
       2737.3987, 2736.997 , 2739.4204, 2740.5972, 2766.71  , 2779.2659,
       2788.209 , 2767.2375, 2751.92  , 2770.5107, 2766.8071, 2778.158 ,
       2774.783 , 3105.5825, 3120.2246, 3117.9092, 3125.4927, 3120.8086,
       3089.641 , 3077.2368, 3091.6934, 3086.2197, 3107.1523, 3055.1516,
       3070.6724, 3064.271 , 3068.8804, 3046.0925, 3054.7705, 3057.2126,
       3061.7747, 3052.0212, 3051.0542, 2980.8706, 2975.1274, 2966.227 ,
       2962.6555, 2945.4082, 2947.878 , 2961.5571, 2962.7124, 2978.6343,
       2966.1787, 2970.3845, 2971.6626, 2960.2668, 2958.4053, 2914.3833,
       2922.728 , 2934.978 , 2928.3462, 2916.3696, 

In [29]:
print(y_dev)

0     4161.4
1     1836.4
2     3970.1
3     2509.8
4     4952.0
       ...  
94    2865.5
95    2860.7
96    2902.6
97    3000.7
98    2054.5
Name: PullTest (N), Length: 99, dtype: float64


# Check Validation Data

In [30]:
# Category array (must be aligned with y_dev)
categories = dev_df.groupby("Sample ID")["Category"].first().values

plot_visualizer(
    true_vals=y_dev,
    pred_vals=predictions,
    categories=categories,
    title=f"Validation Samples: True vs Prediction ({model}) by Category - Data-Driven Training"
)

# Check Validation Loss and R2

In [31]:

# Calculate MAE and RMSE and R2
mae  = mean_absolute_error(y_dev, predictions)
rmse = np.sqrt(root_mean_squared_error(y_dev, predictions))**2
R2   = r2_score(y_dev, predictions)



print(f"MAE:  {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R2: {R2:.2f}")


MAE:  127.40
RMSE: 219.69
R2: 0.62


# Maybe Feature Importance?


In [32]:
#feature_names = x_train.columns.tolist()
# Fix: tell SHAP what the model's feature names are 
#regressor.feature_names_in_ = np.array(feature_names)

# Calculate SHAP values
#shap_values = interpretability.shap.get_shap_values(
#    estimator=regressor,
#    test_x=test_x,
#    attribute_names=feature_names,
#    algorithm="permutation",
#)

# Create visualization
#fig = interpretability.shap.plot_shap(shap_values)

In [33]:
#x_dev.columns


# Cross Validation

In [34]:
cross_df = pd.read_csv("data/train_dev_data.csv")

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

mae_list = []
rmse_list = []
R2_list = []

for fold, (train_index, val_index) in enumerate(skf.split(cross_df["Sample ID"], cross_df["Category"])):
    x_tr, y_tr = get_features_and_target(cross_df.iloc[train_index], target_column) 
    x_val, y_val = get_features_and_target(cross_df.iloc[val_index], target_column)

    x_tr_scale = sc.fit_transform(X=x_tr)
    x_val_scale = sc.transform(x_val)

    if model == 'XGBoost':
        dtrain = xgb.DMatrix(x_tr_scale, label=y_tr)
        dval = xgb.DMatrix(x_val_scale, label=y_val)

        params = {
            'objective': 'reg:squarederror',
            'max_depth': 1,
            'eta': 0.57,
            'eval_metric': 'rmse'
        }

        num_boost_round = 20
        bst = xgb.train(params, dtrain, num_boost_round)

        preds = bst.predict(dval)

    elif model == 'RandomForest':

        params = {
            'n_estimators': 11,
            'max_depth': 5,
            'min_samples_split': 2,
            'min_samples_leaf': 4,
            'random_state': 42,
        }

        preds = RandomForestRegressor(**params).fit(x_tr_scale, y_tr).predict(x_val_scale)
        
    else:
        regressor = TabPFNRegressor()
        regressor.fit(x_tr, y_tr)

        preds = regressor.predict(x_val)

    mae  = mean_absolute_error(y_val, preds)
    rmse = root_mean_squared_error(y_val, preds)
    R2   = r2_score(y_val, preds)

    mae_list.append(mae) 
    rmse_list.append(rmse) 
    R2_list.append(R2)

    # Categories
    categories= cross_df.iloc[val_index]["Category"].values

    plot_visualizer(
        true_vals=y_val,
        pred_vals=preds,
        categories=categories,
        title=f"Fold {fold+1}: True vs Prediction ({model}) by Category - Cross-Validation Data-Driven Training"
    )

    print(f"\nFold {fold+1}")
    print("MAE :", mae)
    print("RMSE:", rmse)
    print("R²  :", R2)

mae_mean = np.mean(mae_list) 
rmse_mean = np.mean(rmse_list) 
R2_mean = np.mean(R2_list) 



Fold 1
MAE : 135.182265772964
RMSE: 236.60780058596063
R²  : 0.6137761464990084



Fold 2
MAE : 134.05336243141699
RMSE: 260.9522233838478
R²  : 0.6883020093296474



Fold 3
MAE : 132.29876438752387
RMSE: 216.10244323990707
R²  : 0.6818142374020981


In [35]:
print(f"mean MAE:  {mae_mean:.2f}")
print(f"mean RMSE: {rmse_mean:.2f}")
print(f"mean R²:   {R2_mean:.2f}")

mean MAE:  133.84
mean RMSE: 237.89
mean R²:   0.66


In [36]:
print(f"mean MAE:  {mae_mean:.2f}")
print(f"mean RMSE: {rmse_mean:.2f}")
print(f"mean R²:   {R2_mean:.2f}")


mean MAE:  133.84
mean RMSE: 237.89
mean R²:   0.66


In [37]:
print(f"mean MAE:  {mae_mean:.2f}")
print(f"mean RMSE: {rmse_mean:.2f}")
print(f"mean R²:   {R2_mean:.2f}")

mean MAE:  133.84
mean RMSE: 237.89
mean R²:   0.66


# Select between XGB and RandomForest

In [69]:
#model_type = "xgbregressor"
model_type = "random_forest" 

# Model Creation

In [70]:
def create_model(model_type, params):
    if model_type == "xgbregressor":
        return XGBRegressor(
            objective='reg:squarederror',
            max_depth=params['max_depth'],
            learning_rate=params['learning_rate'],
            n_estimators=params['n_estimators'],
            eval_metric='rmse'
        )
    elif model_type == "random_forest":
        return RandomForestRegressor(
            n_estimators=params['n_estimators'],
            max_depth=params['max_depth'],
            min_samples_split=params['min_samples_split'],
            min_samples_leaf=params['min_samples_leaf'],
            random_state=42,
        )


# Parameter Selection

In [ ]:
if model_type == "xgbregressor":
    param_grid = []
    max_depth_values = range(1, 5)
    eta_values = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7] # first search [0.5, 0.51, 0.52, 0.53, 0.54, 0.55, 0.56, 0.57]
    n_estimators_values = range(50, 201, 50)

    for md in max_depth_values:
        for eta in eta_values:
            for ne in n_estimators_values:
                param_grid.append({
                    'max_depth': md,
                    'learning_rate': eta,
                    'n_estimators': ne
                })

elif model_type == "random_forest":
    param_grid = []
    n_estimators_values = [17] # first search range(100, 1001, 100) #range(10, 101, 10)
    max_depth_values = [11]
    min_samples_split_values = [2] # first search range(2, 11)
    min_samples_leaf_values = [5]

    for ne in n_estimators_values:
        for md in max_depth_values:
            for mss in min_samples_split_values:
                for msl in min_samples_leaf_values:
                    param_grid.append({
                        'n_estimators': ne,
                        'max_depth': md,
                        'min_samples_split': mss,
                        'min_samples_leaf': msl
                    })


# Gridsearch

In [74]:
best_rmse = float("inf")
best_params = None
best_model = None

for params in param_grid:
    print(f"\nTesting params: {params}")

    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

    mae_list = []
    rmse_list = []
    R2_list = []

    for fold, (train_index, val_index) in enumerate(
            skf.split(cross_df["Sample ID"], cross_df["Category"])):

        X_tr, y_tr = get_features_and_target(cross_df.iloc[train_index], target_column)
        X_val, y_val = get_features_and_target(cross_df.iloc[val_index], target_column)

        X_tr_scale = sc.fit_transform(X_tr)
        X_val_scale = sc.transform(X_val)

        model = create_model(model_type, params)
        
        model.fit(X_tr_scale, y_tr)
        preds = model.predict(X_val_scale)

        mae = mean_absolute_error(y_val, preds) 
        rmse = root_mean_squared_error(y_val, preds) 
        R2 = r2_score(y_val, preds) 

        mae_list.append(mae) 
        rmse_list.append(rmse) 
        R2_list.append(R2)

    mae_mean = np.mean(mae_list) 
    rmse_mean = np.mean(rmse_list) 
    R2_mean = np.mean(R2_list) 

    if rmse_mean < best_rmse:
        best_rmse = rmse_mean
        best_params = params
        best_model = model

        print("\n==============================")
        print(" BEST MODEL FOUND ")
        print("==============================")
        print(f"Best RMSE:   {best_rmse:.2f}")
        print(f"Best Params: {best_params}")



Testing params: {'n_estimators': 17, 'max_depth': 11, 'min_samples_split': 2, 'min_samples_leaf': 5}

 BEST MODEL FOUND 
Best RMSE:   254.09
Best Params: {'n_estimators': 17, 'max_depth': 11, 'min_samples_split': 2, 'min_samples_leaf': 5}


# RandomSearch

In [ ]:
from scipy.stats import randint
from sklearn.model_selection import RandomizedSearchCV

# use a different variable name to avoid overwriting the notebook-level 'model' string
rf_estimator = RandomForestRegressor()

param_dist = {
    'n_estimators': randint(10, 200),
    'max_depth': randint(1, 20),
    'min_samples_split': randint(2, 11),
    'min_samples_leaf': randint(1, 11)
}

randomized_search = RandomizedSearchCV(
    rf_estimator,
    param_distributions=param_dist,
    n_iter=100,
    cv=5,
    scoring='neg_mean_squared_error',  # regression scoring
    n_jobs=-1,
    random_state=42
)

# fit on the existing training variables (use scaled features if desired)
randomized_search.fit(x_train_scale, y_train)

,estimator,RandomForestRegressor()
,param_distributions,"{'max_depth': <scipy.stats....x7e23b45920e0>, 'min_samples_leaf': <scipy.stats....x7e23b45939a0>, 'min_samples_split': <scipy.stats....x7e23b4592350>, 'n_estimators': <scipy.stats....x7e23b510bac0>}"
,n_iter,100
,scoring,'neg_mean_squared_error'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [121]:
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

model_name = "xgb"  # or "xgbregressor"

def create_regressor(model_name, params):
    if model_name == "rf":
        return RandomForestRegressor(**params, random_state=42)
    elif model_name == "xgb":
        return XGBRegressor(
            **params,
            random_state=42,
            tree_method="hist",
            eval_metric="rmse"
        )
    else:
        raise ValueError(f"Unknown model: {model_name}")



In [131]:
from scipy.stats import randint, uniform

param_spaces = {
    "rf": {
        "n_estimators": randint(50, 300),
        "max_depth": randint(3, 20),
        "min_samples_split": randint(2, 10),
        "min_samples_leaf": randint(1, 10)
    },
    "xgb": {
        'n_estimators' : randint(1, 300),
        'max_depth' : randint(1, 5),
        'learning_rate': randint(1, 11)   #

    }
}

In [132]:
def sample_params(space):
    params = {}
    for k, v in space.items():
        val = v.rvs()

        # Convert learning_rate integer → stepped float
        if k == "learning_rate":
            val = round(val / 10.0, 1)   # 1→0.1, 2→0.2, ..., 10→1.0

        params[k] = val
    return params


In [133]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_absolute_error, r2_score
import numpy as np

def randomized_cv_search(model_name, cross_df, target_column, n_iter):
    space = param_spaces[model_name]

    best_rmse = float("inf")
    best_params = None
    best_model = None

    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

    for i in range(n_iter):
        params = sample_params(space)
        print(f"\nTesting params: {params}")

        mae_list, rmse_list, r2_list = [], [], []

        for fold, (train_index, val_index) in enumerate(
                skf.split(cross_df["Sample ID"], cross_df["Category"])):

            train_df = cross_df.iloc[train_index].copy()
            val_df   = cross_df.iloc[val_index].copy()

            X_tr, y_tr = get_features_and_target(train_df, target_column)
            X_val, y_val = get_features_and_target(val_df, target_column)

            X_tr_scale = sc.fit_transform(X_tr)
            X_val_scale = sc.transform(X_val)

            model = create_regressor(model_name, params)
            model.fit(X_tr_scale, y_tr)

            preds = model.predict(X_val_scale)

            mae = mean_absolute_error(y_val, preds)
            rmse = np.sqrt(np.mean((y_val - preds)**2))
            r2 = r2_score(y_val, preds)

            mae_list.append(mae)
            rmse_list.append(rmse)
            r2_list.append(r2)

        rmse_mean = np.mean(rmse_list)
        print(f"RMSE={rmse_mean:.3f}")

        if rmse_mean < best_rmse:
            best_rmse = rmse_mean
            best_params = params
            best_model = model

    return best_model, best_params, best_rmse


In [ ]:
best_model, best_params, best_rmse = randomized_cv_search(
    model_name="xgb",         # ← SWITCH TO XGBOOST
    cross_df=cross_df,
    target_column="PullTest (N)",
    n_iter=100000
)



Testing params: {'n_estimators': 239, 'max_depth': 2, 'learning_rate': 0.2}
RMSE=275.316

Testing params: {'n_estimators': 53, 'max_depth': 1, 'learning_rate': 0.8}
RMSE=256.827

Testing params: {'n_estimators': 124, 'max_depth': 4, 'learning_rate': 0.2}
RMSE=265.805

Testing params: {'n_estimators': 283, 'max_depth': 3, 'learning_rate': 0.4}
RMSE=273.084

Testing params: {'n_estimators': 52, 'max_depth': 3, 'learning_rate': 0.1}
RMSE=255.658

Testing params: {'n_estimators': 66, 'max_depth': 3, 'learning_rate': 0.9}
RMSE=293.450

Testing params: {'n_estimators': 238, 'max_depth': 4, 'learning_rate': 0.1}
RMSE=260.293

Testing params: {'n_estimators': 8, 'max_depth': 1, 'learning_rate': 0.6}
RMSE=249.760

Testing params: {'n_estimators': 169, 'max_depth': 2, 'learning_rate': 0.2}
RMSE=271.770

Testing params: {'n_estimators': 173, 'max_depth': 1, 'learning_rate': 0.5}
RMSE=256.212

Testing params: {'n_estimators': 171, 'max_depth': 3, 'learning_rate': 0.3}
RMSE=272.618

Testing params

In [129]:
print("\n==============================")
print(" BEST MODEL FOUND ")
print("==============================")
print(f"Best RMSE:   {best_rmse:.2f}")
print(f"Best Params: {best_params}")

#==============================
# BEST MODEL FOUND XGBoost
#==============================
#Best RMSE:   241.86
#Best Params: {'n_estimators': 15, 'max_depth': 1, 'learning_rate': 0.6}
#Best RMSE:   245.36
#Best Params: {'max_depth': 1, 'learning_rate': 0.2, 'n_estimators': 50}
#Best RMSE:   257.09
#Best Params: {'n_estimators': 28, 'max_depth': 4, 'learning_rate': np.float64(0.10519775309146498)}
#Best RMSE:   263.07
#Best Params: {'n_estimators': 29, 'max_depth': 1, 'learning_rate': 0.5478342030460285}

#==============================
# BEST MODEL FOUND RandomForest
#==============================
#Best RMSE:   229.32
#Best Params: {'n_estimators': 19, 'max_depth': 5, 'min_samples_split': 6, 'min_samples_leaf': 3}
#Best RMSE:   230.34
#Best Params: {'n_estimators': 19, 'max_depth': 7, 'min_samples_split': 9, 'min_samples_leaf': 3}
#Best RMSE:   230.57
#Best Params: {'n_estimators': 20, 'max_depth': 6, 'min_samples_split': 10, 'min_samples_leaf': 3}
#Best RMSE:   234.09
#Best Params: {'n_estimators': 100, 'max_depth': 10, 'min_samples_split': 7, 'min_samples_leaf': 3}
#Best RMSE:   254.09
#Best Params: {'n_estimators': 17, 'max_depth': 11, 'min_samples_split': 2, 'min_samples_leaf': 5}


 BEST MODEL FOUND 
Best RMSE:   241.86
Best Params: {'n_estimators': 15, 'max_depth': 1, 'learning_rate': 0.6}
